In [1]:
import dspy

# Konfiguration des lokalen Sprachmodells
local_llm = dspy.LM(
    "openai/Qwen3-VL-8B-Instruct-Q4_K_M.gguf", 
    api_base="http://localhost:8080/v1", 
    api_key="no_key_needed",
    temperature=2,
    cache=False
)

dspy.configure(lm=local_llm)

In [2]:
def sentiment_accuracy(gold, pred, trace=None):
    """
    Berechnet die Genauigkeit für die Sentiment-Klassifizierung.
    Gibt True zurück, wenn die Vorhersage mit dem Gold-Label übereinstimmt, andernfalls False.
    
    Args:
        gold (dspy.Example): Das Beispiel mit der korrekten Antwort.
        pred (dspy.Prediction): Die Vorhersage des Modells.
        trace (optional): Der Ausführungstrace. Wird hier nicht verwendet.
        
    Returns:
        bool: True bei Übereinstimmung, andernfalls False.
    """
    # Zugriff auf die relevanten Felder in den gold- und pred-Objekten
    gold_sentiment = gold.sentiment
    predicted_sentiment = pred.sentiment
    
    # Normalisierung und Vergleich der Werte (Groß-/Kleinschreibung ignorieren)
    return gold_sentiment.lower() == predicted_sentiment.lower()

In [3]:
# Signatur und Modul (aus Tag 8/9)
class SentimentSignature(dspy.Signature):
    """Klassifiziert den Sentiment eines gegebenen Textes als positiv, negativ oder neutral."""
    text = dspy.InputField(desc="Der zu klassifizierende Text.")
    sentiment = dspy.OutputField(desc="Das Ergebnis der Klassifizierung: positiv, negativ oder neutral.")

class SimpleClassifier(dspy.Module):
    def __init__(self):
        super().__init__()
        self.predictor = dspy.Predict(SentimentSignature)

    def forward(self, text):
        return self.predictor(text=text)

# Definition der Beispieldaten
raw_data = [
    ("Die Bildqualität ist hervorragend und die Bedienung intuitiv.", "positiv"),
    ("Ich bin sehr zufrieden mit dem Produkt, es übertrifft meine Erwartungen.", "positiv"),
    ("Das Preis-Leistungs-Verhältnis ist unschlagbar.", "positiv"),
    ("Leider hat das Gerät nach kurzer Zeit den Geist aufgegeben.", "negativ"),
    ("Der Kundenservice war überhaupt nicht hilfreich und unfreundlich.", "negativ"),
    ("Die Akkulaufzeit ist enttäuschend kurz.", "negativ"),
    ("Das Produkt wurde pünktlich geliefert.", "neutral"),
    ("Die Verpackung war angemessen.", "neutral"),
]
dspy_examples = [dspy.Example(text=t, sentiment=s).with_inputs("text") for t, s in raw_data]

In [9]:
dspy_examples

[Example({'text': 'Die Bildqualität ist hervorragend und die Bedienung intuitiv.', 'sentiment': 'positiv'}) (input_keys={'text'}),
 Example({'text': 'Ich bin sehr zufrieden mit dem Produkt, es übertrifft meine Erwartungen.', 'sentiment': 'positiv'}) (input_keys={'text'}),
 Example({'text': 'Das Preis-Leistungs-Verhältnis ist unschlagbar.', 'sentiment': 'positiv'}) (input_keys={'text'}),
 Example({'text': 'Leider hat das Gerät nach kurzer Zeit den Geist aufgegeben.', 'sentiment': 'negativ'}) (input_keys={'text'}),
 Example({'text': 'Der Kundenservice war überhaupt nicht hilfreich und unfreundlich.', 'sentiment': 'negativ'}) (input_keys={'text'}),
 Example({'text': 'Die Akkulaufzeit ist enttäuschend kurz.', 'sentiment': 'negativ'}) (input_keys={'text'}),
 Example({'text': 'Das Produkt wurde pünktlich geliefert.', 'sentiment': 'neutral'}) (input_keys={'text'}),
 Example({'text': 'Die Verpackung war angemessen.', 'sentiment': 'neutral'}) (input_keys={'text'})]

In [4]:
# Aufteilung des Datensatzes
split_index = int(len(dspy_examples) * 0.8)  # 80/20-Split

trainset = dspy_examples[:split_index]
devset = dspy_examples[split_index:]

# Anzeige der Größe der erstellten Sets
print(f"Anzahl der Beispiele im Trainingsset: {len(trainset)}")
print(f"Anzahl der Beispiele im Evaluationsset: {len(devset)}")

Anzahl der Beispiele im Trainingsset: 6
Anzahl der Beispiele im Evaluationsset: 2


In [6]:
from dspy.teleprompt import BootstrapFewShot

# Metrik für die Optimierung
def metric(gold, pred, trace=None):
    return gold.sentiment.lower() == pred.sentiment.lower()

# Konfiguration und Ausführung des Optimizers
optimizer = BootstrapFewShot(metric=metric, max_bootstrapped_demos=2)
optimized_classifier = optimizer.compile(SimpleClassifier(), trainset=trainset)

# Ausführung mit dem Testtext
test_text = "Der Stoff fühlt sich angenehm an, aber der Schnitt ist seltsam."
optimized_classifier(text=test_text)


 33%|████████████████████████████                                                        | 2/6 [00:01<00:02,  1.87it/s]


Bootstrapped 2 full traces after 2 examples for up to 1 rounds, amounting to 2 attempts.


Prediction(
    sentiment='neutral'
)

In [7]:
from dspy.evaluate import Evaluate

# Instanziierung des Evaluators
evaluator = Evaluate(devset=devset, num_threads=1, display_progress=True)

# Ausführung der Evaluation mit der benutzerdefinierten Metrik
evaluation_score = evaluator(optimized_classifier, metric=sentiment_accuracy)

Average Metric: 2.00 / 2 (100.0%): 100%|█████████████████████████████████████████████████| 2/2 [00:03<00:00,  1.62s/it]

2025/11/13 12:07:12 INFO dspy.evaluate.evaluate: Average Metric: 2 / 2 (100.0%)
